## Preprocesamiento de los datos

Antes de realizar el preprocesamiento, el conjunto de datos se divide aleatoriamente en un 80 % para entrenamiento y un 20 % para prueba. El conjunto de entrenamiento se utiliza para la selección de variables y el ajuste del modelo, mientras que el conjunto de prueba se reserva exclusivamente para evaluar el desempeño final sobre datos no observados durante el entrenamiento.

In [ ]:
# Cargamos los paquetes necesarios ACORDARNOS DE HACER EL AMBIENTE VIRTUAL
import polars as pl
import numpy as np
import pyprojroot
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [9]:
# Definir la ruta raiz del proyecto
ROOT = pyprojroot.here()

# Cargamos los datos
datos_entrenamiento = pl.read_parquet(ROOT / "datos" / "temporada1.parquet")

In [ ]:

datos_pd = datos_entrenamiento.to_pandas()

train_pd, test_pd = train_test_split(
    datos_pd,
    test_size=0.20,
    random_state=42,
    stratify=datos_pd["swing"]
)

train = pl.from_pandas(train_pd)
test = pl.from_pandas(test_pd)

#### Tratamiento de datos faltantes

In [11]:
# Identificamos que variables tienen valores nulos y la cantidad en cada una
na = (
    train
    .null_count()
    .transpose(include_header=True)
    .filter(pl.col("column_0") > 0)
)

na

column,column_0
str,u32
"""release_speed""",303
"""pitch_type""",303
"""pfx_x""",2446
"""pfx_z""",862
"""plate_x""",303
"""plate_z""",333
"""sz_top""",303
"""sz_bot""",342
"""altura_zona""",342


In [12]:
# Se seleccionan las observaciones que contienen al menos un valor nulo
datos_con_na = train.filter(
    pl.any_horizontal(pl.all().is_null())
)

datos_con_na.height

3072

In [13]:
# Se cuenta la cantidad de valores nulos por fila y resumimos cuántas observaciones tienen
train.with_columns(
    pl.sum_horizontal(pl.all().is_null()).alias("NA_por_fila")
).filter(
    pl.col("NA_por_fila") > 0
).group_by("NA_por_fila").len().sort("NA_por_fila")

NA_por_fila,len
u32,u32
1,2728
2,41
9,303


In [14]:
# Se muestra las observaciones que presentan 9 o más valores nulos
train.with_columns(
    pl.sum_horizontal(pl.all().is_null()).alias("NA_por_fila")
).filter(
    pl.col("NA_por_fila") >= 9
)

release_speed,description,stand,p_throws,pitch_type,balls,strikes,pfx_x,pfx_z,plate_x,plate_z,sz_top,sz_bot,altura_zona,swing,NA_por_fila
f64,str,str,str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,u32
null,"""swinging_strike_blocked""","""L""","""R""",null,0,2,null,null,null,null,null,null,null,1,9
null,"""hit_into_play""","""L""","""R""",null,0,2,null,null,null,null,null,null,null,1,9
null,"""foul""","""R""","""R""",null,2,2,null,null,null,null,null,null,null,1,9
null,"""hit_into_play""","""R""","""L""",null,0,2,null,null,null,null,null,null,null,1,9
null,"""called_strike""","""R""","""R""",null,2,2,null,null,null,null,null,null,null,0,9
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,"""hit_into_play""","""L""","""R""",null,1,2,null,null,null,null,null,null,null,1,9
null,"""swinging_strike""","""R""","""R""",null,3,2,null,null,null,null,null,null,null,1,9
null,"""hit_into_play""","""R""","""L""",null,1,2,null,null,null,null,null,null,null,1,9



Se realizó un análisis de los valores faltantes presentes en el conjunto de entrenamiento (train). Se identificaron 3.072 observaciones con al menos un valor ausente sobre un total de 567.881 registros, lo que representa aproximadamente un 0,54% del conjunto de entrenamiento. Los faltantes se concentran principalmente en variables relacionadas con las características físicas y espaciales de los lanzamientos, tales como la velocidad (`release_speed`), el tipo de lanzamiento (`pitch_type`), el movimiento de la pelota (`pfx_x` y `pfx_z`) y la ubicación del lanzamiento respecto de la zona de strike (`plate_x`, `plate_z`, `sz_top`, `sz_bot` y `altura_zona`).

Si bien la proporción de observaciones incompletas es reducida, la consigna del trabajo requiere generar una predicción para cada lanzamiento presente en `temporada2.parquet`. En consecuencia, la eliminación de registros con valores faltantes podría ocasionar una pérdida de observaciones durante la etapa de validación y dificultar la generación de predicciones para la totalidad de los lanzamientos.

Por este motivo, se opta por imputar los valores faltantes. Para las variables numéricas se utiliza la mediana calculada sobre los datos de entrenamiento. En el caso de las variables categóricas, los valores ausentes se reemplazan por la categoría más frecuente observada en el conjunto de entrenamiento, es decir, la moda.

### Imputación de valores faltantes



Luego del análisis realizado, se aplica la estrategia de imputación seleccionada sobre los conjuntos de entrenamiento y validación. Para evitar la incorporación de información proveniente del conjunto de validación, los parámetros de imputación se calculan únicamente utilizando los datos correspondientes a `train` y posteriormente se aplican sobre ambos conjuntos.

In [15]:
variables_cuantitativas = [
    "release_speed",
    "balls",
    "strikes",
    "pfx_x",
    "pfx_z",
    "plate_x",
    "plate_z",
    "sz_top",
    "sz_bot",
    "altura_zona"
]

variables_categoricas = [
    "pitch_type",
    "stand",
    "p_throws"
]

In [ ]:

# Imputador para variables numéricas
imputador_num = SimpleImputer(strategy="median")

# Se ajusta con train
train[variables_cuantitativas] = imputador_num.fit_transform(
    train[variables_cuantitativas]
)

# Se aplica a test
test[variables_cuantitativas] = imputador_num.transform(
    test[variables_cuantitativas]
)

# Imputación de variables categóricas
# Para que funcione imputador_cat se convirtió el dataframe de polars a uno de pandas

# Conversión a pandas
train = train.to_pandas()
test = test.to_pandas()

# Imputador para variables categóricas
imputador_cat = SimpleImputer(strategy="most_frequent")

# Se calcula la moda en train
train[variables_categoricas] = imputador_cat.fit_transform(
    train[variables_categoricas]
)

# Se aplica la misma moda a test
test[variables_categoricas] = imputador_cat.transform(
    test[variables_categoricas]
)


Una vez realizada la imputación, se chequea que ya no haya valores nulos presentes en los dataframes de entrenamiento y validación.

In [20]:
# Confirmación de la no existencia de valores nulos en las variables categoricas
train[variables_categoricas].isna().sum()

pitch_type    0
stand         0
p_throws      0
dtype: int64

In [21]:
test[variables_categoricas].isna().sum()

pitch_type    0
stand         0
p_throws      0
dtype: int64

In [1]:
# Confirmación de la no existencia de valores nulos en las variables cuantitativas
train[variables_cuantitativas].isna().sum()

NameError: name 'train' is not defined

In [23]:
test[variables_cuantitativas].isna().sum()

release_speed    0
balls            0
strikes          0
pfx_x            0
pfx_z            0
plate_x          0
plate_z          0
sz_top           0
sz_bot           0
altura_zona      0
dtype: int64

In [24]:
# Volvemos el DataFrame a polars
train = pl.from_pandas(train)
test = pl.from_pandas(test)

In [25]:
# Guardamos el conjunto de entrenamiento imputado
train.write_parquet("../datos/train.parquet")
test.write_parquet("../datos/test.parquet")

### Tratamiento de valores atípicos (Outliers)

Luego de realizar las imputaciones correspondientes, se analizan los valores atípicos de las variables presentes en el conjunto de datos. Para ello, se utilizará como referencia el análisis descriptivo realizado previamente, el cual permitió comprender las características de las variables. A partir de esta información, se evaluará si los valores atípicos identificados requieren algún tratamiento.

Para continuar, se comparó el análisis descriptivo previo con los límites reales del béisbol. Esto permitió detectar dos tipos de anomalías:
- **Errores reglamentarios**: Un caso aislado donde la variable `balls` tomó el valor de 4 (lo máximo permitido en juego es 3).
- **Lanzamientos extremos**: Valores atípicos en las variables continuas, como la velocidad del lanzamiento (`release_speed`) de 30 mph (lo que indica lanzamientos muy lentos), posición vertical de la pelota (`plate_z`) negativas de -5.07 pies (lo que significa que la pelota picó en el suelo) y posición horizontal de la pelota (`plate_x`) de más de 9 pies (lo que indica que las pelotas se fueron totalmente hacia los costados).

In [26]:
# Se define las condiciones de lo que consideramos "mal" 
condicion_error = (
    (pl.col("balls") > 3) |                       
    (pl.col("release_speed") < 60) |            
    (pl.col("plate_z") < -2) | (pl.col("plate_z") > 7) |  
    (pl.col("plate_x") < -4) | (pl.col("plate_x") > 4)   
)

A partir de estas condiciones, se creó un conjunto de datos denominado `datos_sospechosos`. Este subconjunto agrupa todas las observaciones que presentaron alguna de las condiciones antes mencionadas.

El objetivo es cuantificar cuántos datos representan respecto al total del conjunto de entrenamiento. De esta manera, poder evaluar su impacto porcentual y tomar una decisión fundamentada sobre si corresponde eliminarlos o aplicar otro tipo de tratamiento.

In [27]:
# Se crea el dataset de observaciones sospechosas
datos_sospechosos = train.filter(condicion_error)

# Se calculan los números para el análisis
total_filas = train.height
cantidad_malos = datos_sospechosos.height
porcentaje_malos = (cantidad_malos / total_filas) * 100

print(f"Total de registros en el dataset: {total_filas}")
print(f"Cantidad de filas con outliers: {cantidad_malos}")
print(f"Porcentaje que representan: {porcentaje_malos:.3f}%")

Total de registros en el dataset: 567881
Cantidad de filas con outliers: 335
Porcentaje que representan: 0.059%


El anterior análisis arrojó un total de 335 observaciones sospechosas sobre los 567881 registros. Esto significa que los valores atípicos representan apenas el $0.059\%$ del total del conjunto de entrenamiento.

Esta proporción refleja que, si bien existen anomalías físicas y reglamentarias en la captura de los lanzamientos, la cantidad es tan chica que demuestra que estos errores son casos muy aislados y no afectan al resto de los datos.

A partir de los resultados obtenidos, se decidió conservar estas 335 observaciones dentro del conjunto de entrenamiento en lugar de procede. Al representar una cantidad muy pequeña, mantener estos registros permite preservar la integridad del conjunto de datos original sin introducir sesgos ni distorsionar el comportamiento de los futuros modelos predictivos. Asimismo, esta decisión permite reflejar la realidad de los datos, donde conviven jugadas reales y extremas junto con pequeños errores aislados en los sistemas de medición de los estadios.